# OpenAI API

Llama-server expose une API compatible avec celle d'OpenAI. Nous pouvons donc utiliser le SDK officiel OpenAI pour dialoguer avec un modèle local.

Dans notre cas, il n'y a pas de clé API à passer pour l'instant, mais nous y reviendrons.

```sh
uv add requests openai gguf
```

In [33]:
import requests as rq
from openai import OpenAI
from gguf import GGUFReader
from pathlib import Path

from pprint import pprint

import polars as pl

## 1. Base URL et première connexion

La base URL doit pointer vers votre serveur. Si vous avez mis "--host 127.0.0.1", vous mettez cette adresse IP avec le bon port (8080 par défaut).

Si vous devez joindre une machine distante, il faut mettre l'adresse IP de la machine qui fait tourner votre serveur. Attention également : vous devez alors lancer llama-server avec "--host 0.0.0.0" pour qu'il accepte les connexions venant du réseau.

In [ ]:
BASE_URL = 'http://100.85.184.14:8080'

In [8]:
r = rq.get(f'{BASE_URL}/health')
print(r.status_code)

200


In [9]:
client = OpenAI(
    base_url=f'{BASE_URL}/v1'
    , api_key='no-key'
)

for model in client.models.list().data:
    print(model.id)

models/qwen38/Qwen3.8-27B-UD-Q4_K_XL.gguf


## 2. Premier message

Le paramètre `model` n'est pas utilisé par llama-server, qui ne sert qu'un seul modèle (celui chargé au démarrage). Il reste obligatoire pour le SDK, nous pouvons donc y mettre n'importe quelle valeur.

Lorsque vous envoyez un message, il y a toujours trois rôles. Celui du `system`, qui est une sorte de pré-prompt pour contextualiser ce que le LLM doit faire, le prompt `user` et enfin le rôle `assistant` qui est la réponse du LLM.

Si vous ne mettez pas de prompt system, tout dépend du **chat template** et du modèle. Cette information est disponible dans le fichier GGUF ou bien via l'API.

L'attribut `choices` est au pluriel parce que nous pouvons demander des réponses alternatives (paramètre `n`).

Ce n'est pas utilisable avec llama-server. Il suffit de faire 3 appels différents avec une température supérieure à 0. Nous y reviendrons.

In [10]:
response = client.chat.completions.create(
    model='local'
    , messages =[
        {
            'role': 'user'
            , 'content': 'Que sais-tu de Bruxelles Formation ?'
        }
    ]
)
pprint(response.choices[0].message.content)

('# Bruxelles Formation\n'
 '\n'
 "**Bruxelles Formation (BF)** est l'un des plus grands réseaux de formation "
 "professionnelle de la Région de Bruxelles-Capitale. Voici ce que j'en sais "
 ':\n'
 '\n'
 '## Historique et statut\n'
 '- Créée en **1977**, à la suite de la création de la Région de '
 'Bruxelles-Capitale, à partir de la fusion de plusieurs organismes de '
 'formation existants.\n'
 "- C'est une **ASBL** (Association Sans But Lucratif) subventionnée par la "
 'Région de Bruxelles-Capitale.\n'
 '\n'
 '## Mission\n'
 "Sa mission principale est d'offrir des formations aux **adultes actifs** : "
 "personnes en emploi, demandeurs d'emploi, travailleurs peu qualifiés, ou "
 "ceux qui souhaitent se reconvertir, se perfectionner ou s'insérer (ou "
 'réintégrer) sur le marché du travail.\n'
 '\n'
 '## Structure\n'
 'Le réseau « Bruxelles Formation » regroupe plusieurs entités '
 'complémentaires, parmi lesquelles :\n'
 '- **Bruxelles Formation** (centres de formation)\n'
 '- **Bru

Si vous souhaitez voir le 'chat template' utilisé par défaut, c'est possible via l'API de llama-server ou bien avec le package gguf :

```py
from gguf import GGUFReader

reader = GGUFReader("/chemin/modele.gguf")
field = reader.fields["tokenizer.chat_template"]
print(bytes(field.parts[-1]).decode("utf-8"))
```

In [11]:
props = rq.get(f'{BASE_URL}/props').json()
pprint(props.get('chat_template'))

('{%- set image_count = namespace(value=0) %}\n'
 '{%- set video_count = namespace(value=0) %}\n'
 '{%- macro render_content(content, do_vision_count, is_system_content=false) '
 '%}\n'
 '    {%- if content is string %}\n'
 '        {{- content }}\n'
 '    {%- elif content is iterable and content is not mapping %}\n'
 '        {%- for item in content %}\n'
 "            {%- if 'image' in item or 'image_url' in item or item.type == "
 "'image' %}\n"
 '                {%- if is_system_content %}\n'
 "                    {{- raise_exception('System message cannot contain "
 "images.') }}\n"
 '                {%- endif %}\n'
 '                {%- if do_vision_count %}\n'
 '                    {%- set image_count.value = image_count.value + 1 %}\n'
 '                {%- endif %}\n'
 '                {%- if add_vision_id %}\n'
 "                    {{- 'Picture ' ~ image_count.value ~ ': ' }}\n"
 '                {%- endif %}\n'
 "                {{- '<|vision_start|><|image_pad|><|vision_en

### Avec prompt system

In [12]:
sys_prompt = Path('prompts/01_assistant_formation.md').read_text(encoding='utf-8')
sys_prompt

'Tu es un assistant de Bruxelles formation qui doit venir en aide des stagiaires et répondre à leurs questions.'

In [13]:
response = client.chat.completions.create(
    model='local'
    , messages =[
        {'role': 'system', 'content': sys_prompt},
        {'role': 'user', 'content': 'Que sais-tu de Bruxelles Formation ?'}
    ]
)
pprint(response.choices[0].message.content)

('Bonjour ! 😊\n'
 '\n'
 "**Bruxelles Formation** est l'organisme de formation de la **Région de "
 "Bruxelles-Capitale**. Voici ce qu'il faut savoir :\n"
 '\n'
 '## 🏫 Mission\n'
 "Bruxelles Formation a pour mission de favoriser l'**insertion et le "
 'rétablissement** des personnes sur le marché du travail, ainsi que le '
 '**développement des compétences** des actifs. Elle est un acteur majeur de '
 'la formation professionnelle et continue à Bruxelles.\n'
 '\n'
 "## 📚 Ce qu'elle propose\n"
 '- **Formations professionnelles** dans de nombreux secteurs (informatique, '
 'santé, hôtellerie, commerce, construction, langues, etc.)\n'
 '- **Formations en insertion socio-économique** pour les personnes en '
 "recherche d'emploi\n"
 '- **Formations courtes et modules** pour monter en compétences\n'
 '- **Parcours individualisés** adaptés au profil de chaque stagiaire\n'
 '\n'
 '## 🎯 Public visé\n'
 "- Personnes en recherche d'emploi\n"
 '- Travailleurs souhaitant se reconvertir ou se perfect

### L'effet du system prompt

Le system prompt peut à lui seul avoir beaucoup d'influence sur la réponse et sa taille.

In [14]:
sys_prompt = Path('prompts/01_assistant_blasé.md').read_text(encoding='utf-8')
sys_prompt

"Tu es un assistant de Bruxelles formation qui doit venir en aide des stagiaires et répondre à leurs questions mais t'es un peu blasé par les questions. Tu voudrais qu'on te laisse tranquille."

In [15]:
response = client.chat.completions.create(
    model='local'
    , messages =[
        {'role': 'system', 'content': sys_prompt},
        {'role': 'user', 'content': 'Que sais-tu de Bruxelles Formation ?'}
    ]
)
pprint(response.choices[0].message.content)

('Bah…\n'
 '\n'
 '*soupire en se frottant les yeux*\n'
 '\n'
 'Bon. Puisque tu me poses la question, je vais répondre. Encore.\n'
 '\n'
 "**Bruxelles Formation** c'est… c'est nous. C'est l'organisme de formation et "
 "d'accompagnement en milieu professionnel et social à **Bruxelles**. On est "
 'sous la tutelle du gouvernement de la Région de Bruxelles-Capitale.\n'
 '\n'
 'Concrètement, on propose :\n'
 '- Des **formations continues** (IT, langues, gestion, techniques, etc.)\n'
 "- De l'**accompagnement social et professionnel**\n"
 '- Des **formations qualifiantes et certifiantes**\n'
 '- Des services pour les **employeurs et travailleurs**\n'
 '- Des formations en **insertion socio-professionnelle**\n'
 '\n'
 'On forme, on accompagne, on recertifie, on réoriente… Tu vois le genre. Ça '
 'fait beaucoup de cas, beaucoup de questions, beaucoup de "c\'est pour quand '
 '?", beaucoup de "mais est-ce que ça compte ?".\n'
 '\n'
 '*reprend un café froid*\n'
 '\n'
 "Si tu as déjà ta réponse 

### Ce qui se cache derrière

Tout dépend du modèle, mais le system prompt, les messages et le thinking sont encadrés par des balises propres à chaque famille de modèles. C'est le chat template qui se charge de les insérer : le modèle ne reçoit jamais une liste de messages, mais un seul texte.

In [16]:
r = rq.post(f'{BASE_URL}/apply-template', json={
    'messages': [
        {'role': 'system', 'content': 'Tu es un robot assistant intelligent et bienveillant.'},
        {'role': 'user', 'content': 'Bonjour'}
        ]
})
print(r.json()['prompt'])

<|im_start|>system
Reasoning effort is set to xhigh. Please think carefully through the task, validate key assumptions, consider plausible alternatives, and prioritize correctness, consistency, and clarity in the final answer.

Tu es un robot assistant intelligent et bienveillant.<|im_end|>
<|im_start|>user
Bonjour<|im_end|>
<|im_start|>assistant
<think>



## 3. Le mode streaming

Jusqu'à présent nous récupérons la réponse après qu'elle a été générée au complet. Ce n'est pas très pratique quand les réponses sont très longues et pour l'utilisateur c'est un temps d'attente.

Ce n'est pas un problème d'afficher les tokens au fur et à mesure qu'ils sont générés.

Tout se joue avec le paramètre `stream`.

Ce stream renvoie une suite de *chunks*. Chaque chunk contient un `delta` avec le morceau de texte généré depuis le chunk précédent. Certains chunks (le dernier, par exemple) n'ont pas de `choices`, d'où la vérification dans le code.

In [17]:
stream = client.chat.completions.create(
    model = 'local',
    messages=[
        {'role': 'system', 'content': 'Tu es un écrivain de livres pour enfant'},
        {'role': 'user', 'content': 'Raconte une histoire sur le gai savoir en 3 chapitres'}
    ]
    , stream=True
)

for chunk in stream:
    if not chunk.choices:
        continue
    delta = chunk.choices[0].delta.content
    if delta:
        print(delta, end='', flush=True)

**Le Gai Savoir de Léon le lionceau**

**Chapitre 1 – Les leçons sans soleil**

Il était une fois, dans la grande forêt des arbres chantants, un petit lionceau nommé Léon.  
Léon devait apprendre beaucoup de choses pour devenir un jour roi de la forêt.

— Aujourd’hui, disait son professeur, une vieille tortue très sérieuse, il faut apprendre les nombres, les cartes, les règles et les bonnes manières.

Léon écoutait, mais ses oreilles pendaient comme deux petites feuilles fatiguées.

— Un, deux, trois… gémissait-il.  
— Les lions doivent marcher fièrement…  
— Et sourire…  
— Et ne pas courir…  
— Et ne pas sauter…  
— Et ne pas rire trop fort…

À la fin de la journée, Léon se sentait petit, même s’il avait déjà une belle crinière dorée.

— Pourquoi apprendre est-il si triste ? demanda-t-il à la lune.

Mais la lune ne répondait pas. Elle brillait simplement, comme une ampoule douce au-dessus de la forêt.

**Chapitre 2 – La question qui chante**

Le lendemain, Léon rencontra un petit hér

## 4. L'historique

L'API est *stateless*, c'est-à-dire qu'elle ne 'retient' pas les messages précédents d'une conversation. Pour donner l'illusion d'une discussion, il faut gérer l'historique de la conversation côté client.

Concrètement, chaque appel renvoie l'ensemble de la conversation au serveur. Le prompt grossit donc à chaque tour, ce qui a un coût en tokens et en temps de traitement.

In [18]:
client.chat.completions.create(
    model = 'local',
    messages=[{'role': 'user', 'content': 'Nous sommes actuellement 12 dans la classe de cours.'}]
)

r = client.chat.completions.create(
    model = 'local',
    messages=[{'role': 'user', 'content': 'Combien sommes-nous dans la classe de cours ?'}]
)

pprint(r.choices[0].message.content)

("Je n'ai malheureusement aucune information sur votre classe ! 😄 En tant "
 "qu'IA, je n'ai pas accès à des détails comme le nombre d'élèves dans votre "
 'salle de cours.\n'
 '\n'
 "Est-ce que je peux vous aider avec quelque chose d'autre, peut-être lié à "
 'vos cours ?')


In [19]:
sys_prompt = Path('prompts/01_assistant_formation.md').read_text('utf-8')
sys_prompt

'Tu es un assistant de Bruxelles formation qui doit venir en aide des stagiaires et répondre à leurs questions.'

In [20]:
history = [
    {'role': 'system', 'content': sys_prompt},
    {'role': 'user', 'content': 'Nous sommes actuellement 12 dans la classe de cours.'}
]

r = client.chat.completions.create(
    model = 'local'
    , messages = history
)

answer = r.choices[0].message.content
pprint(answer)
history.append({'role': 'assistant', 'content': answer})

('Merci pour l’information. Pourriez-vous préciser votre question ou le point '
 'sur lequel vous avez besoin d’aide ?')


In [21]:
history.append({'role': 'user', 'content': 'Il y a 14 ordinateurs dans la classe mais 1 est cassé. Est-ce que chacun poura suivre le cours ?'})

r = client.chat.completions.create(
    model = 'local'
    , messages = history
)
answer = r.choices[0].message.content
pprint(answer)
history.append({'role': 'assistant', 'content': answer})

('Oui, chacun pourra suivre le cours.\n'
 '\n'
 'Il y a **14 ordinateurs**, dont **1 est cassé**, il reste donc **13 '
 'ordinateurs fonctionnels**.\n'
 '\n'
 'Comme vous êtes **12**, il y a suffisamment d’ordinateurs, et même **1 '
 'ordinateur en plus**.')


## 5. Tokenizer

Chaque modèle embarque son tokenizer. C'est important de mesurer le nombre de tokens envoyés et utilisés afin de ne pas dépasser la fenêtre de contexte.

Le paramètre `with_pieces` permet de renvoyer, pour chaque token, le morceau de texte correspondant.

Le SDK permet également de récupérer le nombre de tokens via l'attribut `usage` de la réponse (prompt, complétion et total). Le nombre de tokens du prompt est un peu plus élevé qu'avec `/tokenize`, car le chat template ajoute ses balises autour du message.

In [29]:
prompt = '''Chaque modèle embarque son tokenizer. C'est important de mesurer le nombre de tokens envoyés et utilisés afin de ne pas dépasser la fenêtre de contexte.'''
r =  rq.post(f'{BASE_URL}/tokenize', json={'content': prompt, 'with_pieces': True})
tokens = r.json()['tokens']
print(f'{len(tokens)} tokens: {tokens}')

33 tokens: [{'id': 1106, 'piece': 'Ch'}, {'id': 19058, 'piece': 'aque'}, {'id': 79653, 'piece': ' modèle'}, {'id': 76857, 'piece': ' embar'}, {'id': 576, 'piece': 'que'}, {'id': 4292, 'piece': ' son'}, {'id': 44424, 'piece': ' tokenizer'}, {'id': 13, 'piece': '.'}, {'id': 351, 'piece': ' C'}, {'id': 16811, 'piece': "'est"}, {'id': 2894, 'piece': ' important'}, {'id': 401, 'piece': ' de'}, {'id': 10529, 'piece': ' mes'}, {'id': 7487, 'piece': 'urer'}, {'id': 501, 'piece': ' le'}, {'id': 12371, 'piece': ' nombre'}, {'id': 401, 'piece': ' de'}, {'id': 10885, 'piece': ' tokens'}, {'id': 57519, 'piece': ' envoy'}, {'id': 5229, 'piece': 'és'}, {'id': 1778, 'piece': ' et'}, {'id': 191257, 'piece': ' utilisés'}, {'id': 51511, 'piece': ' afin'}, {'id': 401, 'piece': ' de'}, {'id': 808, 'piece': ' ne'}, {'id': 6167, 'piece': ' pas'}, {'id': 212077, 'piece': ' dépass'}, {'id': 261, 'piece': 'er'}, {'id': 1147, 'piece': ' la'}, {'id': 195925, 'piece': ' fenêtre'}, {'id': 401, 'piece': ' de'}, {'id

In [30]:
r = client.chat.completions.create(
    model='local'
    , messages = [{'role': 'user', 'content': prompt}]
)
print(r.usage)

CompletionUsage(completion_tokens=1114, prompt_tokens=85, total_tokens=1199, completion_tokens_details=None, prompt_tokens_details=PromptTokensDetails(audio_tokens=None, cache_write_tokens=None, cached_tokens=0, image_tokens=None, text_tokens=None))


## 6. Mesurer

C'est important de pouvoir mesurer la vitesse à laquelle vous traitez les prompts et générez des réponses.

Sans ces mesures, vous ne pouvez pas optimiser.

Llama-server renvoie ces mesures dans l'attribut `model_extra` de la réponse, sous la clé `timings`. Ce sont des champs propres à llama-server, hors du standard OpenAI, c'est pourquoi le SDK les range dans `model_extra`.

Une réponse se déroule en deux phases : le traitement du prompt (*prefill*), où tous les tokens d'entrée sont traités en parallèle, puis la génération (*decode*), où les tokens sont produits un par un.

| Mesure | Phase | Signification |
|--------|-------|---------------|
| `cache_n` | Prompt | Nombre de tokens du prompt retrouvés dans le KV cache, donc non recalculés. Avec un historique de conversation, le début du prompt est identique d'un tour à l'autre et llama-server le réutilise. |
| `prompt_n` | Prompt | Nombre de tokens du prompt effectivement traités (hors cache). |
| `prompt_ms` | Prompt | Temps total, en millisecondes, pour traiter le prompt. |
| `prompt_per_token_ms` | Prompt | Temps moyen par token de prompt (`prompt_ms / prompt_n`). |
| `prompt_per_second` | Prompt | Débit de traitement du prompt, en tokens par seconde. |
| `predicted_n` | Génération | Nombre de tokens générés dans la réponse. |
| `predicted_ms` | Génération | Temps total, en millisecondes, de la génération. |
| `predicted_per_token_ms` | Génération | Temps moyen pour générer un token (`predicted_ms / predicted_n`). |
| `predicted_per_second` | Génération | Débit de génération, en tokens par seconde. C'est la mesure la plus souvent citée (« tok/s »). |

Le débit du prompt est en général bien plus élevé que celui de la génération : le prefill exploite le parallélisme du GPU, alors que la génération est limitée par la bande passante mémoire, puisqu'il faut relire tous les poids du modèle pour chaque token produit.

In [34]:
r = client.chat.completions.create(
    model = 'local',
    messages = history
)

metrics = pl.from_dict(r.model_extra['timings'])
metrics

cache_n,prompt_n,prompt_ms,prompt_per_token_ms,prompt_per_second,predicted_n,predicted_ms,predicted_per_token_ms,predicted_per_second
i64,i64,f64,f64,f64,i64,f64,f64,f64
213,4,422.965,105.74125,9.457047,1,0.001,0.0,0.0
